# Парсинг валют и драгметаллов

Алгоритм:
1. Заходите на страницу с ценами: `https://www.moex.com/ru/issue/GLDRUB_TOM/CETS`
2. Берете оттуда engine (движок), market (рынок), board (режим торгов) и тикер
3. Вставляете в функцию `get_moex_data`
4. ???
5. Profit!

Для драг. металлов и валюты:
- engine: `currency`
- market: `selt`
- board: `cets`




In [60]:
from enum import Enum, IntEnum
from aiomoex import get_board_candles
import asyncio
import aiohttp
from datetime import datetime, timedelta
import pandas as pd


class Engines(Enum):
    """https://iss.moex.com/iss/engines"""

    STOCK = "stock"  # Фондовый рынок и рынок депозитов
    STATE = "state"  # Рынок ГЦБ (размещение)
    CURRENCY = "currency"  # Валютный рынок
    FUTURES = "futures"  # stockСрочный рынок
    COMMODITY = "commodity"  # Товарный рынок
    INTERVENTIONS = "interventions"  # Товарные интервенции
    OFFBOARD = "offboard"  # ОТС-система
    AGR = "agro"  # Агро
    OTC = "otc"  # ОТС с ЦК
    QUOTES = "quotes"  # Квоты
    MONEY = "money"  # Денежный рынок


class Markets(Enum):
    """https://iss.moex.com/iss/engines/<engine>/markets"""

    OTCINDICES = "otcindices"  # Внебиржевые индексы
    SELT = "selt"  # Биржевые сделки с ЦК
    FUTURES = "futures"  # Поставочные фьючерсы
    INDEX = "index"  # Валютный фиксинг
    OTC = "otc"  # Внебиржевой


class Boards(Enum):
    """https://iss.moex.com/iss/engines/<engine>/markets/<market>/boards"""

    TQBR = "TQBR"  # Фондовый рынок
    AUCB = "AUCB"  # Аукцион ЦБР - адрес.
    CETS = "CETS"  # Системные сделки - безадрес.
    CNGD = "CNGD"  # Внесистемные сделки- адрес.
    CURR = "CURR"  # Дневная сессия
    FIXN = "FIXN"  # Фиксинг внесистемный- адрес.
    FIXS = "FIXS"  # Фиксинг системный - безадрес.
    LICU = "LICU"  # Внесистемные сделки урегулирования - безадрес.
    SDBP = "SDBP"  # Крупные сделки - безадрес.
    SPEC = "SPEC"  # Поставка - безадресные
    WAPN = "WAPN"  # Внесистемные средневзвешенные - адрес.
    WAPS = "WAPS"  # Системные средневзвешенные - безадрес.


class IntervalEnum(IntEnum):
    MINUTE = 1
    TEN_MINUTES = 10
    HOUR = 60
    DAY = 24
    WEEK = 7
    MONTH = 31


# объявим аннотацию для удобства
StockData = list[dict[str, str | int | float]]


async def fetch_ticker_data(
    session: aiohttp.ClientSession,
    ticker: str,
    interval: IntervalEnum,
    start_date: str,
    end_date: str,
    board: Boards,
    engine: Engines,
    market: Markets,
) -> dict[str, StockData]:
    """Функция получает данные о торгах по заданному тикеру с *start_date* по *end_date* с интервалом *interval*, возвращая словарь, где ключом является тикер, а значением - данные

    Args:
        session (aiohttp.ClientSession): aiottp сессия для отпаравки запросов
        ticker (str): Имя тикера
        interval (IntervalEnum): Одно из доступных значений для интервала времени
        start_date (str): Начальная дата в формате yyyy-mm-dd
        end_date (str): Конечная дата в формате yyyy-mm-dd

    Returns:
        dict[str, list[dict[str, str | int | float]]]: Словарь, где ключ - тикер, а значение - данные, например {'SBER': sber_data}
    """
    try:
        print(f"Запрашиваем данные для {ticker}...")
        # получаем данные по переданному тикеру за указанный период
        res = await get_board_candles(
            session,
            ticker,
            interval,
            start_date,
            end_date,
            board=board.value,
            market=market.value,
            engine=engine.value,
        )
        print(f"Получено {len(res)} записей для {ticker}")
        return {ticker: res}
    except Exception as e:
        print(f"Ошибка парсинга. Не удалось получить данные для {ticker}: {e}")
        print(f"Тип ошибки: {type(e).__name__}")
        return {ticker: []}


async def get_moex_data(
    tickers: list[str],
    start_date: datetime,
    end_date: datetime = datetime.now(),
    interval: IntervalEnum = IntervalEnum.DAY,
    engine: Engines = Engines.CURRENCY,
    market: Markets = Markets.SELT,
    board: Boards = Boards.CETS,
) -> dict[str, StockData]:
    if interval not in IntervalEnum:
        raise ValueError(f"Неверный интервал. Допустимые значения: {IntervalEnum}")

    end_date_formatted = end_date.strftime("%Y-%m-%d")
    start_date_formatted = start_date.strftime("%Y-%m-%d")
    
    print(f"Запрашиваем данные для тикеров: {tickers}")
    print(f"Период: {start_date_formatted} - {end_date_formatted}")
    print(f"Параметры: engine={engine.value}, market={market.value}, board={board.value}")

    # Увеличиваем таймауты для MOEX API
    timeout = aiohttp.ClientTimeout(
        connect=30,      # время подключения
        sock_read=60,   # время чтения данных
        total=120       # общий таймаут
    )
    
    async with aiohttp.ClientSession(timeout=timeout) as session:
        # собираем корутины в список
        coros = [
            fetch_ticker_data(
                session,
                ticker,
                interval,
                start_date_formatted,
                end_date_formatted,
                board,
                engine,
                market,
            )
            for ticker in tickers
        ]

        print("Отправляем запросы к MOEX API...")
        # 'собираем' результаты корутин - непосредственно парсинг
        stock_data = await asyncio.gather(*coros)
        print("Получены ответы от MOEX API")

    # разворачиваем список словарей в один словарь, например: [ {'SBER': sber_data}, {'GAZP': gazp_data} ] -> { 'SBER': sber_data, 'GAZP': gazp_data }
    stock_data = {
        ticker: data for element in stock_data for ticker, data in element.items()
    }
    
    # Выводим информацию о полученных данных
    for ticker, data in stock_data.items():
        print(f"Тикер {ticker}: получено {len(data)} записей")
        if data:
            print(f"Первая запись: {data[0]}")
    
    return stock_data

In [61]:
from dataclasses import dataclass


@dataclass
class Asset:
    """Общий класс для активов с нужными параметрами для получения данных с MOEX"""
    engine: Engines
    market: Markets
    board: Boards
    ticker: str

    async def get_candles(self, start_date: datetime, end_date: datetime, interval: IntervalEnum) -> pd.DataFrame:
        data =  await get_moex_data(
            tickers=[self.ticker],
            start_date=start_date,
            end_date=end_date,
            interval=interval,
            engine=self.engine,
            market=self.market,
            board=self.board,
        )
        df = pd.DataFrame(data[self.ticker])
        df["ticker"] = self.ticker
        return df

In [42]:
gold = Asset(
    engine=Engines.CURRENCY,
    market=Markets.SELT,
    board=Boards.CETS,
    ticker="GLDRUB_TOM",
)
display(await gold.get_candles(start_date=datetime.now() - timedelta(days=180), end_date=datetime.now(), interval=IntervalEnum.DAY))

Запрашиваем данные для тикеров: ['GLDRUB_TOM']
Период: 2025-04-29 - 2025-10-26
Параметры: engine=currency, market=selt, board=CETS
Отправляем запросы к MOEX API...
Запрашиваем данные для GLDRUB_TOM...
Получено 126 записей для GLDRUB_TOM
Получены ответы от MOEX API
Тикер GLDRUB_TOM: получено 126 записей
Первая запись: {'open': 8701.1, 'close': 8662.1, 'high': 8719.9, 'low': 8630, 'value': 2250719190, 'volume': 259709, 'begin': '2025-04-29 00:00:00', 'end': '2025-04-29 23:59:59'}


,open,close,high,low,value,volume,begin,end,ticker
0,8701.1,8662.1,8719.9,8630.0,2.250719e+09,259709,2025-04-29 00:00:00,2025-04-29 23:59:59,GLDRUB_TOM
1,8678.0,8610.3,8709.8,8560.0,2.141582e+09,248664,2025-04-30 00:00:00,2025-04-30 23:59:59,GLDRUB_TOM
2,8560.0,8599.8,8690.0,8556.3,6.763267e+08,78401,2025-05-02 00:00:00,2025-05-02 23:59:59,GLDRUB_TOM
3,8672.5,8675.0,8684.9,8599.9,1.570787e+09,181738,2025-05-05 00:00:00,2025-05-05 23:59:59,GLDRUB_TOM
4,8756.5,8802.0,8824.9,8741.0,2.238773e+09,254910,2025-05-06 00:00:00,2025-05-06 23:59:59,GLDRUB_TOM
...,...,...,...,...,...,...,...,...,...
121,10986.0,11207.0,11220.6,10955.0,6.948383e+09,627937,2025-10-20 00:00:00,2025-10-20 23:59:59,GLDRUB_TOM
122,11251.0,10880.0,11289.9,10660.0,7.543267e+09,687506,2025-10-21 00:00:00,2025-10-21 23:59:59,GLDRUB_TOM
123,11000.0,10529.0,11037.0,10490.0,9.297243e+09,869267,2025-10-22 00:00:00,2025-10-22 23:59:59,GLDRUB_TOM
124,10750.0,10850.0,10850.0,10609.6,5.016714e+09,468023,2025-10-23 00:00:00,2025-10-23 23:59:59,GLDRUB_TOM


In [59]:
usdrub = Asset(
    engine=Engines.CURRENCY,
    market=Markets.SELT,
    board=Boards.CETS,
    ticker="USD000UTSTOM",
)
display(await usdrub.get_candles(start_date=datetime.now() - timedelta(days=180), end_date=datetime.now(), interval=IntervalEnum.DAY))

Запрашиваем данные для тикеров: ['USD000UTSTOM']
Период: 2025-04-29 - 2025-10-26
Параметры: engine=currency, market=selt, board=CETS
Отправляем запросы к MOEX API...
Запрашиваем данные для USD000UTSTOM...
Получено 0 записей для USD000UTSTOM
[]
Получены ответы от MOEX API
Тикер USD000UTSTOM: получено 0 записей
{'USD000UTSTOM': []}


,ticker


In [53]:
from aiomoex import request_helpers
print(request_helpers.make_url(
    engine=Engines.CURRENCY.value,
    market=Markets.SELT.value,
    board=Boards.CETS.value,
    security="USD000UTSTOM",
))
print(request_helpers.make_query(interval=IntervalEnum.DAY, start='2025-01-01', end='2025-10-26'))

https://iss.moex.com/iss/engines/currency/markets/selt/boards/CETS/securities/USD000UTSTOM.json
{'interval': <IntervalEnum.DAY: 24>, 'from': '2025-01-01', 'till': '2025-10-26'}


In [33]:
gold_ticker = "GLDRUB_TOM"

tickers = [gold_ticker]

delta = timedelta(days=180)
start_date = datetime.now() - delta

stock_data = await get_moex_data(
    tickers,
    start_date=start_date,
    interval=IntervalEnum.DAY,
    engine=Engines.CURRENCY,
    market=Markets.SELT,
    board=Boards.CETS,
)

gold_df = pd.DataFrame(stock_data[gold_ticker])
gold_df["ticker"] = gold_ticker
gold_df

Запрашиваем данные для тикеров: ['GLDRUB_TOM']
Период: 2025-04-29 - 2025-10-26
Параметры: engine=currency, market=selt, board=CETS
Отправляем запросы к MOEX API...
Запрашиваем данные для GLDRUB_TOM...
Получено 126 записей для GLDRUB_TOM
Получены ответы от MOEX API
Тикер GLDRUB_TOM: получено 126 записей
Первая запись: {'open': 8701.1, 'close': 8662.1, 'high': 8719.9, 'low': 8630, 'value': 2250719190, 'volume': 259709, 'begin': '2025-04-29 00:00:00', 'end': '2025-04-29 23:59:59'}


,open,close,high,low,value,volume,begin,end,ticker
0,8701.1,8662.1,8719.9,8630.0,2.250719e+09,259709,2025-04-29 00:00:00,2025-04-29 23:59:59,GLDRUB_TOM
1,8678.0,8610.3,8709.8,8560.0,2.141582e+09,248664,2025-04-30 00:00:00,2025-04-30 23:59:59,GLDRUB_TOM
2,8560.0,8599.8,8690.0,8556.3,6.763267e+08,78401,2025-05-02 00:00:00,2025-05-02 23:59:59,GLDRUB_TOM
3,8672.5,8675.0,8684.9,8599.9,1.570787e+09,181738,2025-05-05 00:00:00,2025-05-05 23:59:59,GLDRUB_TOM
4,8756.5,8802.0,8824.9,8741.0,2.238773e+09,254910,2025-05-06 00:00:00,2025-05-06 23:59:59,GLDRUB_TOM
...,...,...,...,...,...,...,...,...,...
121,10986.0,11207.0,11220.6,10955.0,6.948383e+09,627937,2025-10-20 00:00:00,2025-10-20 23:59:59,GLDRUB_TOM
122,11251.0,10880.0,11289.9,10660.0,7.543267e+09,687506,2025-10-21 00:00:00,2025-10-21 23:59:59,GLDRUB_TOM
123,11000.0,10529.0,11037.0,10490.0,9.297243e+09,869267,2025-10-22 00:00:00,2025-10-22 23:59:59,GLDRUB_TOM
124,10750.0,10850.0,10850.0,10609.6,5.016714e+09,468023,2025-10-23 00:00:00,2025-10-23 23:59:59,GLDRUB_TOM


In [14]:
# Тестируем с долларом для проверки работоспособности API
print("\n=== Тест 4: Проверяем работоспособность с долларом ===")
usd_ticker = "USDRUB_TOM"

usd_data = await get_moex_data(
    tickers=[usd_ticker],
    start_date=start_date,
    interval=IntervalEnum.DAY,
    engine=Engines.CURRENCY,
    market=Markets.SELT,
    board=Boards.CETS,
)

if usd_data[usd_ticker]:
    usd_df = pd.DataFrame(usd_data[usd_ticker])
    usd_df["ticker"] = usd_ticker
    print(f"USD: получено {len(usd_df)} записей")
    print(usd_df.head())
    print("API работает корректно!")
else:
    print("Проблема с API или параметрами")



=== Тест 4: Проверяем работоспособность с долларом ===
Запрашиваем данные для тикеров: ['USDRUB_TOM']
Период: 2025-10-19 - 2025-10-26
Параметры: engine=currency, market=selt, board=CETS
Отправляем запросы к MOEX API...
Запрашиваем данные для USDRUB_TOM...
Получено 0 записей для USDRUB_TOM
Получены ответы от MOEX API
Тикер USDRUB_TOM: получено 0 записей
Проблема с API или параметрами
